# Auditoria e preparação — Caminho 2

Objetivo: classificar o índice registrado em baixo/médio/alto por município/dia com meteorologia anterior. A limpeza e a consolidação foram concluídas. Este notebook apresenta os registros auditados da preparação e verifica a base entregue; para refazer a leitura dos quatro CSVs, use `scripts/prepare_course.py` e `scripts/build_course_panel.py` conforme o README. Não é previsão física de ocorrência.

In [1]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'submission/manifest.json').is_file() and (p/'src/fire_risk').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Abra o notebook dentro da pasta extraída ou do clone completo do projeto.')
sys.path.insert(0, str(ROOT/'src'))
import pandas as pd
from IPython.display import display
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
from fire_risk.submission import verify_submission, training_correlations
panel, manifest = verify_submission(ROOT)
def evidence(name):
    return json.loads((ROOT/'submission/evidence'/name).read_text(encoding='utf-8'))
print('Base consolidada íntegra:', len(panel), 'municípios/dia; partições:', manifest['splits'])
print('Modelo congelado:', manifest['model_id'])


No event loop hook running.
Base consolidada íntegra: 1329 municípios/dia; partições: {'train': 764, 'test': 310, 'validation': 183, 'gap': 72}
Modelo congelado: 852606b93da31ade


## Fontes e limpeza rastreável

As quatro fontes do curso estão identificadas por hash. A preparação compara todas as colunas, remove repetições exatas somente dos derivados e reserva horários com versões divergentes. Não há imputação de chuva ausente como zero. Chuva exige 24 registros; temperatura, umidade e vento, ao menos 18. O manifesto da fonte e o resumo abaixo são evidências da execução original, não uma nova leitura dos CSVs neste notebook.

In [2]:
sources = evidence('manifest.json')
display(pd.DataFrame(sources['files'])[['file','sha256']])
preparation = evidence('resumo_preparacao.json')
display(pd.Series(preparation, name='valor').to_frame())
assert preparation['meteorologia_linhas_originais'] == (preparation['duplicatas_exatas_removidas_derivado'] + preparation['linhas_distintas_reservadas_por_chave_multipla'] + preparation['linhas_horarias_usadas'])


,file,sha256
0,BDMEP-estacao.csv,2183def179a3ef349fd304b4f4e7aa6a3849ea8138670c...
1,MapBiomas.csv,a902b3d20d00938e00c61ac69568aa034ff22397bfae8f...
2,BDMEP-2025.csv,5505bf0e723a16575f28ba5bfb50dd5632c4e219ea35cd...
3,queimadas_2024_2025.csv,1ba34802bce6fadcac1f3e6d4bd4c2caeea303de82001b...


,valor
meteorologia_linhas_originais,4891176
meteorologia_linhas_distintas,2463184
duplicatas_exatas_removidas_derivado,2427992
chaves_com_multiplas_variantes,11080
linhas_distintas_reservadas_por_chave_multipla,22160
linhas_horarias_usadas,2441024
estacoes_dias,101759
estacoes_dias_precipitacao_24h,56350
estacoes_dias_sem_municipio,992
linhas_deteccoes,8226944


## Consolidação e dicionário

O painel entregue contém todas as colunas descritas em `docs/data-dictionary.md`: chaves, alvo, quatro features, cobertura, disponibilidade e partições. A junção exige município e D−2. Ausência de detecção não recebe rótulo baixo. As contagens permitem rastrear o descarte por índice inválido e por meteorologia não elegível. Os 72 exemplos de intervalo estão preservados e marcados `gap`, sem entrar no ajuste.

In [3]:
summary = evidence('panel-summary.json')
print(pd.Series({k:summary[k] for k in ['municipality_days_with_records','municipality_days_without_valid_index','valid_target_days_without_eligible_weather','rows']}).to_string())
print(panel.groupby('split').agg(exemplos=('target','size'),inicio=('data','min'),fim=('data','max')).to_string())
print('Ausências no painel:', int(panel.isna().sum().sum()))
print('Duplicatas município/dia:', int(panel.duplicated(['id_municipio','data']).sum()))
assert not panel.isna().any().any()
assert not panel.duplicated(['id_municipio','data']).any()


municipality_days_with_records                19985
municipality_days_without_valid_index           779
valid_target_days_without_eligible_weather    17877
rows                                           1329
            exemplos     inicio        fim
split                                     
gap               72 2025-01-26 2025-02-03
test             310 2025-02-04 2025-02-12
train            764 2025-01-03 2025-01-25
validation       183 2025-01-27 2025-02-02
Ausências no painel: 0
Duplicatas município/dia: 0


## Contrato executável e reconstrução opcional

Reaplicamos validação e classificação ao painel. A célula permite refazer a limpeza e junção chamando o pacote, quando os quatro CSVs estiverem disponíveis. Os resultados temporários preservam a avaliação original.

In [4]:
from fire_risk.panel import validate_course_panel, risk_class, build_panel_frame
validate_course_panel(panel)
assert panel['target_index'].map(risk_class).eq(panel['target']).all()
display(panel.head(10))
RAW_DIR = None  # Informe a pasta dos quatro CSVs para reconstruir.
if RAW_DIR is not None:
    import tempfile
    sys.path.insert(0, str(ROOT))
    from preparar_eda_queimadas import prepare
    with tempfile.TemporaryDirectory() as temporary:
        output = Path(temporary)
        prepare(ROOT, raw_dir=Path(RAW_DIR), output_dir=output)
        rebuilt, summary = build_panel_frame(Path(RAW_DIR)/'queimadas_2024_2025.csv', output/'meteorologia_diaria.parquet')
        display(rebuilt.head())


,id_municipio,data,target_index,detection_records,valid_index_records,weather_date,chuva_d2,temperatura_d2,umidade_d2,vento_d2,eligible_stations,target,split,prediction_issued_at,target_window_end,assumed_features_available_at,historical_publication_verified
0,1400100,2025-01-03,0.840,17,17,2025-01-01,0.0,29.587500,55.458333,3.175000,1,alto,train,2025-01-03,2025-01-04,2025-01-03,False
1,1500602,2025-01-03,0.055,4,4,2025-01-01,0.8,25.475000,94.666667,0.825000,1,baixo,train,2025-01-03,2025-01-04,2025-01-03,False
2,1501709,2025-01-03,0.020,6,6,2025-01-01,0.0,28.620833,78.541667,2.687500,1,baixo,train,2025-01-03,2025-01-04,2025-01-03,False
3,1507300,2025-01-03,0.010,1,1,2025-01-01,9.8,26.220833,84.333333,1.412500,1,baixo,train,2025-01-03,2025-01-04,2025-01-03,False
4,1508001,2025-01-03,0.000,1,1,2025-01-01,8.8,26.875000,86.250000,1.000000,1,baixo,train,2025-01-03,2025-01-04,2025-01-03,False
5,1716505,2025-01-03,0.000,2,2,2025-01-01,0.0,25.712500,80.291667,0.150000,1,baixo,train,2025-01-03,2025-01-04,2025-01-03,False
6,2100501,2025-01-03,0.480,2,2,2025-01-01,7.4,25.391667,79.750000,0.562500,1,medio,train,2025-01-03,2025-01-04,2025-01-03,False
7,2103000,2025-01-03,0.330,11,11,2025-01-01,0.0,29.050000,68.458333,0.983333,1,baixo,train,2025-01-03,2025-01-04,2025-01-03,False
8,2103208,2025-01-03,0.030,15,15,2025-01-01,0.0,29.687500,65.666667,2.350000,1,baixo,train,2025-01-03,2025-01-04,2025-01-03,False
9,2112407,2025-01-03,0.305,6,6,2025-01-01,0.0,28.954167,72.833333,3.275000,1,baixo,train,2025-01-03,2025-01-04,2025-01-03,False


## Conclusões e limites

A preparação e o painel estão concluídos. A janela bruta comum é de 43 dias e o painel elegível tem 1.329 linhas. MapBiomas foi auditado e excluído das features porque legenda/coleção e período são incompatíveis. As escalas meteorológicas originais são preservadas; as unidades físicas não foram inferidas. O atraso de publicação de 24 horas é uma hipótese, não comprovação operacional. Modelagem e API constam dos notebooks seguintes. Submissão e aprovação do curso são etapas separadas.